In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/research/fed-learning/code
CWD: /home/james/git/research/fed-learning


In [2]:
import torch
import numpy as np
from torch import nn
import torch.nn.functional as F
from torch.optim import SGD, Adam
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import copy

In [3]:
from src.models.MatrixFactorization import MF, UMF
from src.data_utils import create_batched_dataloaders, create_dataloader

In [4]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float("inf")

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False


def centralized_validate_loop(model, val_loader, optimizer, scheduler=None):
    tbar = (val_loader)
    loss_fn = nn.MSELoss(reduction="sum")
    model.eval()
    total_obs = 0
    total_sum_loss = 0
    with torch.no_grad():
        for idx, (inputs, target) in enumerate(tbar):
            if inputs.ndim == 3:
                inputs = inputs.squeeze(0)
                target = target.squeeze(0)
            n_obs = inputs.shape[0]
            score = model(inputs[:, 0], inputs[:, 1])

            sum_loss = loss_fn(score, target.float()).detach().numpy()
            total_sum_loss += sum_loss
            total_obs += n_obs

        avg_loss = np.sqrt(total_sum_loss / total_obs)
        if scheduler is not None:
            scheduler.step(avg_loss)

    return avg_loss


def centralized_train_loop(model, train_loader, optimizer):
    loss_fn = nn.MSELoss(reduction="mean")
    model.train()
    total_n_obs = 0
    total_sum_loss = 0
    tbar = tqdm(train_loader)
    ten_percent = len(train_loader) // 10
    for idx, (inputs, target) in enumerate(tbar):
        if inputs.ndim == 3:
            inputs = inputs.squeeze(0)
            target = target.squeeze(0)
        n_obs = inputs.shape[0]
        optimizer.zero_grad()
        score = model(inputs[:, 0], inputs[:, 1])
        loss = loss_fn(score, target.float())
        loss.backward()  # Calculate Gradients
        optimizer.step()  # Current user's update

        total_n_obs += n_obs
        total_sum_loss += loss.detach().numpy() * n_obs
        avg_mse_loss = total_sum_loss / total_n_obs
        if idx % ten_percent == 0:
            tbar.set_description(f"Training Loss: {np.sqrt(avg_mse_loss):.05f}")
            if np.isnan(avg_mse_loss):
                print("Training Loss is NaN")
                raise ValueError
    return total_sum_loss / total_n_obs



In [5]:
train_df = pd.read_csv("dataset/ml100k_train.csv")
test_df = pd.read_csv("dataset/ml100k_test.csv")
n_users = train_df.iloc[:, 0].nunique()
n_items = train_df.iloc[:, 1].nunique()
print(f"{n_users=}, {n_items=}")

n_users=943, n_items=1628


In [6]:
train_df, val_df = train_test_split(train_df, test_size=.2, stratify=train_df.user_id, random_state=0)

In [7]:
train_df

,user_id,item_id,rating
73402,58,566,5
50527,263,1066,4
37207,91,12,4
29688,473,482,5
73441,403,877,3
...,...,...,...
17829,209,14,4
51180,158,258,4
58083,233,544,1
41314,53,0,4


In [8]:
train_dl = create_dataloader(train_df, dl_type="centralized", batch_size=1)
val_dl = create_dataloader(val_df, dl_type="centralized", batch_size=100)
test_dl = create_dataloader(test_df, dl_type="centralized", batch_size=1000)

In [9]:
# model = MF(n_users, n_items, n_factors=30)
model = UMF(n_items, n_factors=30)

In [10]:
early_stopper = EarlyStopper(patience=2)
# def loss_fn(preds, y):
#     return torch.sqrt(F.mse_loss(preds, y))
loss_fn = nn.MSELoss()
# optimizer = Adam(model.parameters(), lr=.01)
optimizer = SGD(model.parameters(), lr=.01, weight_decay=.001)#, momentum=.5)

In [11]:
for i in range(20):
    model.train()
    centralized_train_loop(model, train_dl, optimizer)
    model.eval()
    val_loss = centralized_validate_loop(model, val_loader=val_dl, optimizer=optimizer)
    print(f"val loss: {val_loss}")
    if early_stopper.early_stop(val_loss):
        print("Stopping early")
        break

  0%|          | 0/60000 [00:00<?, ?it/s]

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not int

In [63]:
centralized_validate_loop(model, val_loader=test_dl, optimizer=optimizer)

1.1416207249564458

In [65]:
# torch.save(model.state_dict(), "cent-mf-ml100k.pth")

In [66]:
tmp_model = MF(n_users, n_items, n_factors=30)
tmp_model.load_state_dict(torch.load("cent-mf-ml100k.pth"))

<All keys matched successfully>

In [ ]:
train_dl = create_dataloader(train_df, dl_type="centralized", batch_size=1)
for idx, (X, y) in enumerate(train_dl):
    print(X, X.shape)
    print(f"{X[:,0].shape}")
    print(f"{y=}", y.shape)
    z = y.unsqueeze(-1)
    print(f"{z=}", z.shape)
    preds = model.predict(X[:, 0], X[:, 1])
    print(f"{preds=}", preds.shape)
    loss = loss_fn(preds, z)
    print(f"{loss=}", loss.shape)
    break

In [ ]:
train_dl = create_dataloader(train_df, dl_type="centralized", batch_size=3)
for idx, (X, y) in enumerate(train_dl):
    print(X, X.shape)
    print(f"{y=}", y.shape)
    preds = model.predict(X[:, 0], X[:, 1])
    print(f"{preds=}", preds.shape)
    loss = loss_fn(preds, y)
    print(f"{loss=}", loss.shape)
    break

In [ ]:
loss_fn(y, preds)

In [ ]:
((y-preds)**2).sum() / 3

In [ ]:
((y-preds)**2).mean()

In [ ]:
F.mse_loss(y, preds)

In [ ]:
((y-preds.squeeze())**2).sum() / len(y-preds.squeeze())

In [ ]:
preds.squeeze()